# Transcribe Stacey Boehman Live Coaching Calls (with Speaker Labels)
**Instructions:**
1. Upload your MP3 files to Google Drive in a folder called `coaching_audio`
2. Get a free HuggingFace token at https://huggingface.co/settings/tokens (read access is enough)
3. Accept the pyannote model terms at https://huggingface.co/pyannote/speaker-diarization-3.1
4. Paste your token into Cell 2
5. Run each cell in order
6. Transcripts will be saved to Google Drive in `coaching_transcripts/`
7. Download the `.md` files and upload to Claude Projects

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Set your HuggingFace token
# Get one free at https://huggingface.co/settings/tokens (read access)
# Then accept terms at https://huggingface.co/pyannote/speaker-diarization-3.1
HF_TOKEN = "hf_YOUR_TOKEN_HERE"

In [ ]:
# Cell 3: Install WhisperX
!pip install whisperx -q

In [ ]:
# Cell 4: Transcribe all MP3s with speaker labels
import time
import whisperx
from pathlib import Path

MP3_DIR        = Path('/content/drive/MyDrive/coaching_audio')
TRANSCRIPT_DIR = Path('/content/drive/MyDrive/coaching_transcripts')
TRANSCRIPT_DIR.mkdir(exist_ok=True)

DEVICE      = 'cuda'
BATCH_SIZE  = 16
COMPUTE     = 'float16'
NUM_SPEAKERS = 2  # Stacey + 1 client per call (adjust if group calls)

print('Loading WhisperX large-v3...')
model = whisperx.load_model('large-v3', DEVICE, compute_type=COMPUTE)
print('Model loaded.\n')

mp3s = sorted(MP3_DIR.glob('*.mp3'))
print(f'Found {len(mp3s)} MP3s in {MP3_DIR}\n')

for i, mp3 in enumerate(mp3s, 1):
    out = TRANSCRIPT_DIR / (mp3.stem + '.md')
    if out.exists():
        print(f'[{i}/{len(mp3s)}] SKIP (already done): {mp3.name}')
        continue

    print(f'[{i}/{len(mp3s)}] {mp3.name}')
    t0 = time.time()

    # Step 1: Transcribe
    audio = whisperx.load_audio(str(mp3))
    result = model.transcribe(audio, batch_size=BATCH_SIZE, language='en')
    print(f'  Transcribed in {time.time()-t0:.0f}s')

    # Step 2: Align for word-level timestamps
    align_model, metadata = whisperx.load_align_model(language_code='en', device=DEVICE)
    result = whisperx.align(result['segments'], align_model, metadata, audio, DEVICE)
    print(f'  Aligned in {time.time()-t0:.0f}s')

    # Step 3: Diarize (who spoke when)
    diarize_model = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device=DEVICE)
    diarize_segments = diarize_model(audio, min_speakers=2, max_speakers=NUM_SPEAKERS)
    result = whisperx.assign_word_speakers(diarize_segments, result)
    print(f'  Diarized in {time.time()-t0:.0f}s')

    # Step 4: Identify which speaker is Stacey (speaks the most)
    speaker_word_count = {}
    for seg in result['segments']:
        spk = seg.get('speaker', 'UNKNOWN')
        words = len(seg.get('text', '').split())
        speaker_word_count[spk] = speaker_word_count.get(spk, 0) + words
    stacey_speaker = max(speaker_word_count, key=speaker_word_count.get)
    print(f'  Identified Stacey as {stacey_speaker} ({speaker_word_count})')

    # Step 5: Build labeled transcript
    lines = []
    current_speaker = None
    current_text = []

    for seg in result['segments']:
        spk = seg.get('speaker', 'UNKNOWN')
        label = 'Stacey' if spk == stacey_speaker else 'Client'
        text = seg.get('text', '').strip()
        if not text:
            continue
        if label != current_speaker:
            if current_speaker and current_text:
                lines.append(f'**{current_speaker}:** {" ".join(current_text)}')
            current_speaker = label
            current_text = [text]
        else:
            current_text.append(text)

    if current_speaker and current_text:
        lines.append(f'**{current_speaker}:** {" ".join(current_text)}')

    title = mp3.stem.replace('_', ' ').strip()
    transcript_md = f'# {title}\n\n---\n\n' + '\n\n'.join(lines) + '\n'
    out.write_text(transcript_md, encoding='utf-8')

    elapsed = time.time() - t0
    print(f'  Done in {elapsed:.0f}s — {out.stat().st_size//1024}KB saved to Drive\n')

print(f'All done! Transcripts in {TRANSCRIPT_DIR}')